# 05 · Fair model comparison (Stage 1: screening)

**Goal:** pick the best intent model for Arabic (MSA) + English, fairly, before we commit to one.

**The rules of a fair comparison**
1. Every model gets the **same data**, the **same settings** and the **same seed**. Only the model changes.
2. We choose using the **validation set only**. The test set is opened **once**, at the very end, for the winner. (If we peek at test to choose, our final number is optimistic: that's *test-set leakage*.)
3. We record more than accuracy: macro F1 per language, training time, model size, and **CPU speed** (ms per message), because a real bank server may not have a GPU.

**Plan**
1. Train each candidate once on Arabic + English, same settings.
2. Pick the winner by **validation** macro F1 (average of Arabic and English).
3. Test **only the winner**, once.

**Version 2 (clean validation).** Our first comparison used a validation set that leaked:
Arabic val questions were translations of English *train* questions (and vice versa),
so a multilingual model had partly "seen" the answers. Validation scores were inflated
(about 0.95 on val vs 0.91 on test). Notebook 02 now uses a **group-aware split**
(a question and all its translations stay in the same split), and this notebook reruns
the comparison on that clean validation set. Old results stay in `experiments.csv`;
new ones go to `experiments_clean.csv`.

Runtime → Change runtime type → **T4 GPU** before you start.

## 1 · Check the GPU
Should print `True` and `Tesla T4`.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 2 · Get the code and rebuild the data
Same as notebook 04. `git pull` makes sure you have the latest `src/` code.

In [ ]:
!test -d banking-assistant || git clone https://github.com/YoussefZaky208/banking-assistant.git
%cd /content/banking-assistant
!git pull -q
!pip install -q datasets sentencepiece protobuf
!cd notebooks && python 02_load_and_align_data.py > /dev/null && echo "Data ready"

## 3 · Connect Google Drive
Free Colab can disconnect. Results are saved to your Drive after **each** model, so nothing finished is lost. If Colab disconnects, just run all cells again: finished models are **skipped automatically**.

In [ ]:
# Save results to Google Drive so they survive if Colab disconnects.
# A window asks for permission the first time: allow it.
from google.colab import drive
drive.mount("/content/drive")
RESULTS_DIR = "/content/drive/MyDrive/banking-assistant/results"

## 4 · Load the data
You should see the counts table (ar/en × train/val/test).

In [ ]:
import gc, json, os, shutil, sys, time

import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, EarlyStoppingCallback,
                          Trainer, TrainingArguments, set_seed)

sys.path.append(".")
from src.text_utils import normalize_arabic

os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS_FILE = os.path.join(RESULTS_DIR, "experiments_clean.csv")   # clean validation (no translation leak)
MODELS_DIR = "/content/drive/MyDrive/banking-assistant/models"

data = pd.read_csv("data/banking_en_ar.csv")
label2id = json.load(open("data/labels.json", encoding="utf-8"))
id2label = {i: label for label, i in label2id.items()}
data["label"] = data["intent"].map(label2id)
print(pd.crosstab(data["lang"], data["split"]))

## 5 · How long are the questions? (justifies `max_length = 64`)
Each model's tokenizer splits text into tokens differently (remember **fertility** from notebook 03). If a question is longer than `max_length`, the end gets cut off (**truncation**).
We check: what % of training questions are longer than 64 tokens? If it's around 1% or less, 64 is safe, and it trains about **2× faster** than 128 because attention cost grows with length.

In [ ]:
CANDIDATES = {
    "MARBERTv2": "UBC-NLP/MARBERTv2",
    "AraBERTv2": "aubmindlab/bert-base-arabertv02",
    "XLM-RoBERTa": "FacebookAI/xlm-roberta-base",
}
train_texts = data.loc[data["split"] == "train", ["text", "lang"]]

rows = []
for name, model_id in CANDIDATES.items():
    tok = AutoTokenizer.from_pretrained(model_id)
    lengths = np.array([len(tok(t)["input_ids"]) for t in train_texts["text"]])
    rows.append({"model": name,
                 "median": int(np.median(lengths)),
                 "99th percentile": int(np.percentile(lengths, 99)),
                 "max": int(lengths.max()),
                 "% over 64": round(100 * (lengths > 64).mean(), 2)})
pd.DataFrame(rows)

## 6 · The experiment function
`run_experiment(config)` does everything for one model: tokenize → train with early stopping → score validation per language → time CPU speed → save one row to `experiments.csv` on Drive → delete the model to free GPU memory.

New terms:
- **CPU speed (ms/message):** we classify 200 messages one at a time on the CPU and take the average. That's how a chatbot server actually receives messages.
- **`fp16`:** half-precision numbers, faster on GPU. mDeBERTa is known to break with fp16 (loss becomes **NaN**, "not a number"), so we turn it off for that model only.
- **`gc.collect()` / `empty_cache()`:** clean GPU memory so the next model has room.

In [ ]:
def to_dataset(tok, df, max_length):
    ds = Dataset.from_pandas(df[["text", "label"]], preserve_index=False)
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=max_length), batched=True)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro")}


def cpu_speed(model, tok, texts, max_length):
    """Average milliseconds to classify ONE message on a CPU (like a cheap server)."""
    model = model.to("cpu").eval()
    with torch.no_grad():
        for t in texts[:5]:                      # warm-up, not timed
            model(**tok(t, return_tensors="pt", truncation=True, max_length=max_length))
        start = time.perf_counter()
        for t in texts:
            model(**tok(t, return_tensors="pt", truncation=True, max_length=max_length))
    return 1000 * (time.perf_counter() - start) / len(texts)


def already_done(run_name):
    return os.path.exists(RESULTS_FILE) and run_name in set(pd.read_csv(RESULTS_FILE)["run_name"])


def run_experiment(cfg, evaluate_test=False):
    """Train one model with one set of settings. Decisions use VALIDATION scores only."""
    if already_done(cfg["run_name"]) and not evaluate_test:
        print("Skipping (already in results):", cfg["run_name"])
        return None
    print("\n==========", cfg["run_name"], "==========")
    set_seed(cfg["seed"])
    df = data[data["lang"].isin(cfg["languages"])].copy()
    if cfg.get("normalize"):
        df["text"] = df["text"].map(normalize_arabic)

    tok = AutoTokenizer.from_pretrained(cfg["model_name"])
    train_ds = to_dataset(tok, df[df["split"] == "train"], cfg["max_length"])
    val_df = df[df["split"] == "val"]
    val_ds = to_dataset(tok, val_df, cfg["max_length"])

    model = AutoModelForSequenceClassification.from_pretrained(
        cfg["model_name"], num_labels=len(label2id), id2label=id2label, label2id=label2id)
    out_dir = f"/content/checkpoints/{cfg['run_name']}"
    args = TrainingArguments(
        output_dir=out_dir, learning_rate=cfg["learning_rate"],
        per_device_train_batch_size=cfg["batch_size"], per_device_eval_batch_size=128,
        num_train_epochs=cfg["max_epochs"], eval_strategy="epoch", save_strategy="epoch",
        save_total_limit=1, load_best_model_at_end=True, metric_for_best_model="macro_f1",
        greater_is_better=True, fp16=torch.cuda.is_available() and cfg.get("fp16", True),
        logging_steps=50, report_to="none", seed=cfg["seed"])
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg["patience"])])
    start = time.time()
    trainer.train()
    minutes = (time.time() - start) / 60
    trainer.remove_callback(EarlyStoppingCallback)

    row = {**cfg, "languages": "+".join(cfg["languages"]),
           "epochs": trainer.state.epoch, "train_minutes": round(minutes, 1),
           "params_millions": round(sum(p.numel() for p in model.parameters()) / 1e6)}
    splits = ["val"] + (["test"] if evaluate_test else [])
    for split in splits:
        part_df = df[df["split"] == split]
        for lang in cfg["languages"]:
            sub = part_df[part_df["lang"] == lang]
            m = trainer.evaluate(to_dataset(tok, sub, cfg["max_length"]), metric_key_prefix=split)
            row[f"{split}_{lang}_f1"] = round(m[f"{split}_macro_f1"], 4)
            row[f"{split}_{lang}_acc"] = round(m[f"{split}_accuracy"], 4)
    if any(np.isnan(h.get("loss", 0)) for h in trainer.state.log_history):
        print("WARNING: the loss became NaN. Try this model again with fp16 set to False.")

    row["cpu_ms_per_message"] = round(cpu_speed(model, tok, val_df["text"].tolist()[:200], cfg["max_length"]), 1)
    print({k: v for k, v in row.items() if k.startswith(("val_", "test_", "cpu", "epochs"))})

    if not evaluate_test:   # screening and tuning runs go into the comparison table
        table = pd.DataFrame([row])
        if os.path.exists(RESULTS_FILE):     # add to old rows (columns may differ between runs)
            table = pd.concat([pd.read_csv(RESULTS_FILE), table], ignore_index=True)
        table.to_csv(RESULTS_FILE, index=False)
    if cfg.get("save_to"):   # keep this model for the app
        trainer.save_model(cfg["save_to"])
        tok.save_pretrained(cfg["save_to"])
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    shutil.rmtree(out_dir, ignore_errors=True)
    return row

## 7 · Stage 1: screen the 4 candidates
Same settings for all (the paper's settings, except `max_length` 64, `languages` ar+en).

| Model | Why it's a candidate |
|---|---|
| MARBERTv2 | Arabic model, trained on 1B tweets (dialects + MSA). Best in the paper. |
| AraBERTv2 | Arabic model, trained on news / Wikipedia (MSA). Our Run 1. |
| XLM-RoBERTa base | **Multilingual**: 100 languages incl. Arabic and English. Strong English. |

mDeBERTa-v3 was dropped: its loss was NaN from the first step in our environment, even in full precision with a lower learning rate.

Each model is **saved to Drive**, so the winner doesn't need retraining.
⏱ About **2 hours** total on a T4 (XLM-R alone ~1 hour). You can leave it running.

In [ ]:
BASE = {"languages": ["ar", "en"], "normalize": False, "learning_rate": 4e-5,
        "batch_size": 64, "max_length": 64, "max_epochs": 20, "patience": 5, "seed": 42}

STAGE1 = [
    {**BASE, "run_name": "clean_marbertv2", "model_name": CANDIDATES["MARBERTv2"]},
    {**BASE, "run_name": "clean_arabertv2", "model_name": CANDIDATES["AraBERTv2"]},
    {**BASE, "run_name": "clean_xlmr", "model_name": CANDIDATES["XLM-RoBERTa"]},
]
for cfg in STAGE1:
    cfg["save_to"] = os.path.join(MODELS_DIR, cfg["run_name"])   # keep each model, so the winner needs no retraining
    run_experiment(cfg)

## 8 · The comparison table
Sorted by `val_mean_f1` (average of Arabic and English macro F1). **Send me a screenshot of this table.**

The top row is the winner. It is chosen on validation only, before anyone looks at test.

In [ ]:
results = pd.read_csv(RESULTS_FILE)
results["val_mean_f1"] = results[["val_ar_f1", "val_en_f1"]].mean(axis=1).round(4)
cols = ["run_name", "learning_rate", "seed", "val_ar_f1", "val_en_f1", "val_mean_f1",
        "epochs", "train_minutes", "params_millions", "cpu_ms_per_message"]
results[cols].sort_values("val_mean_f1", ascending=False)

## 9 · Test the winner (the ONE time we use the test set)
Loads the winning model from Drive and scores it on the Arabic and English test sets.
**Send me this table.**

In [ ]:
best = results.sort_values("val_mean_f1", ascending=False).iloc[0]
print("Winner (chosen on VALIDATION only):", best["run_name"])

path = os.path.join(MODELS_DIR, best["run_name"])
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(path)
model = AutoModelForSequenceClassification.from_pretrained(path).to(device).eval()

def predict(texts, batch=128):
    preds = []
    with torch.no_grad():
        for i in range(0, len(texts), batch):
            enc = tok(texts[i:i + batch], padding=True, truncation=True,
                      max_length=int(best["max_length"]), return_tensors="pt").to(device)
            preds += model(**enc).logits.argmax(-1).tolist()
    return preds

final_rows = []
for lang in ["ar", "en"]:
    test = data[(data["split"] == "test") & (data["lang"] == lang)]
    preds = predict(test["text"].tolist())
    final_rows.append({"model": best["run_name"], "lang": lang, "n": len(test),
                       "test_macro_f1": round(f1_score(test["label"], preds, average="macro"), 4),
                       "test_accuracy": round(accuracy_score(test["label"], preds), 4)})
final_test = pd.DataFrame(final_rows)
final_test.to_csv(os.path.join(RESULTS_DIR, "final_test_clean.csv"), index=False)
final_test